In [1]:
import truststore
truststore.extract_from_ssl()

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

########################################################
        INDEXING (OFFLINE SETUP)
########################################################

Data Injestion + Data Cleaning

In [3]:
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from typing import Any, Optional
from enum import Enum
import bs4

class DataType(Enum):
    LINK = "link"
    TEXT = "text"

class DataInjestion:
    def __init__(self, data: Any, type: DataType):
        self.data = data
        self.type = type
        self.text: Optional[list[Document]] = self._load_text()
        self._data_cleaning()

    def _load_text(self):

        loader_map = {
            DataType.LINK: self._link_loader,
            DataType.TEXT: self._text_loader,
        }

        loader = loader_map.get(self.type)
        self.text = loader()

    def _text_loader(self):
        return [Document(self.data)]
    
    def _link_loader(self):
        loader = WebBaseLoader(
            web_path=self.data,
            bs_kwargs=dict(
                parse_only = bs4.SoupStrainer(
                    class_ = ('post-content', 'post-title', 'post-header')
                )
            )
        )

        return loader.load()
    
    def _data_cleaning(self):
        pass

    


/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_41473/134222579.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Chunking

In [7]:
from enum import Enum
from langchain_core.documents import Document
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

class ChunkingType(Enum):
    Fixed_Size = "fixed_size"
    Recursive = "recursive"
    Semantic_Chunking = "semantic_chunking"
    Token_Based = "token_based"
    Hierarchical = "hierarchical"
    Agentic = "agentic"                       # Currently out of scope of project
    Late_Chunking = "late_chunking"

class Chunking:
    def __init__(self, type: ChunkingType, docs: list[Document], **kwargs):
        self.type = type
        self.kwargs = kwargs
        self.docs = docs
        self.chunks = self._chunker(docs)

    def _chunker(self):
        chunking_mapping = {
            ChunkingType.Fixed_Size: self._chunk_fixed_size,
            ChunkingType.Recursive: self._chunk_recursive,
            ChunkingType.Semantic_Chunking: self._chunk_semantic,
            ChunkingType.Token_Based: self._chunk_token_based,
            ChunkingType.Agentic: self._chunk_agentic,
            ChunkingType.Hierarchical: self._chunk_hierarchical,
        }

        return chunking_mapping.get(self.type)
    
    def _chunk_fixed_size(self):
        splitter = CharacterTextSplitter(
            chunk_size = self.kwargs.get("chunk_size", 1000),
            chunk_overlap = self.kwargs.get("chunk_overlap", 50),
            separator = self.kwargs.get("separator", '\n')
        )
        return splitter.split_documents(self.docs)

    def _chunk_recursive(self): 
        splitter = RecursiveCharacterTextSplitter(
            chunk_size = self.kwargs.get("chunk_size", 1000),
            chunk_overlap = self.kwargs.get("chunk_overlap", 50),
            separator = self.kwargs.get("separator", '\n')
        )
        return splitter.split_documents(self.docs)

    def _chunk_semantic(self): 
        splitter = SemanticChunker(
            embeddings=HuggingFaceEmbeddings(),
            breakpoint_threshold_type='percentile',
            breakpoint_threshold_amount=90
        )
        return splitter.split_documents(self.docs)

    def _chunk_token_based(self): 
        splitter = TokenTextSplitter(
            chunk_size = self.kwargs.get("chunk_size", 1000),
            chunk_overlap = self.kwargs.get("chunk_overlap", 50),
        )
        return splitter.split_documents(self.docs)

    def _chunk_hierarchical(self): 
        pass


    def _chunk_agentic(self): 
        pass


Embedding + Indexing